# Data Preprocessing


In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/data_modelo.csv", encoding="latin-1")
df.shape

(50001, 15)

In [17]:
# ── Sentinel replacement ───────────────────────────────────────────────────
df_processed = df.copy()

# Missingness flags for high-null columns before replacing
df_processed["X3_missing"] = (df_processed["X3"] == -9999998).astype(int)
df_processed["X4_missing"] = (df_processed["X4"] == -9999998).astype(int)

SENTINELS = {
    "X3":  [-9999998],
    "X4":  [-9999998],
    "X8":  [999],
    "X11": [-99000720, -99000632],
    "X12": [-99000792],
}

for col, vals in SENTINELS.items():
    for val in vals:
        df_processed[col] = df_processed[col].replace(val, np.nan)

df_processed.isnull().sum()

ID                0
TARGET            0
X1              786
X2             4993
X3            19922
X4            15007
X5            16398
X6                1
X7              551
X8                1
X9              912
X10               0
X11            5836
X12            2394
BASE              0
X3_missing        0
X4_missing        0
dtype: int64

In [18]:
# ── Drop X5 (0.96 correlation with X4) ────────────────────────────────────
df_processed = df_processed.drop(columns=["X5"])

In [19]:
# ── Outlier capping at 99th percentile ────────────────────────────────────
cols_to_cap = ["X1", "X2", "X3", "X4"]

for col in cols_to_cap:
    p99 = df_processed[col].quantile(0.99)
    df_processed[col] = df_processed[col].clip(upper=p99)

In [20]:
# ── Log transformation ─────────────────────────────────────────────────────
# Excludes X6 and X8 (discrete), X9 (categorical), TARGET, ID, BASE, missingness flags
cols_to_log = ["TARGET", "X1", "X2", "X3", "X4", "X10", "X11"]

for col in cols_to_log:
    df_processed[col] = np.log1p(df_processed[col])

In [21]:
df_processed.isnull().sum()

ID                0
TARGET            0
X1              786
X2             4993
X3            19922
X4            15007
X6                1
X7              551
X8                1
X9              912
X10               0
X11            5836
X12            2394
BASE              0
X3_missing        0
X4_missing        0
dtype: int64

In [22]:
# ── Imputation ─────────────────────────────────────────────────────────────
# Median imputation for remaining NaNs in numeric columns
cols_to_impute = ["X1", "X2", "X3", "X4", "X6", "X7", "X8", "X10", "X11", "X12"]

# ── Median imputation (fit on train only) ─────────────────────────────────
train_mask = df_processed["BASE"] == "TRAIN"

imputation_medians = df_processed[train_mask][cols_to_impute].median()
df_processed[cols_to_impute] = df_processed[cols_to_impute].fillna(imputation_medians)

df_processed.isnull().sum()

ID              0
TARGET          0
X1              0
X2              0
X3              0
X4              0
X6              0
X7              0
X8              0
X9            912
X10             0
X11             0
X12             0
BASE            0
X3_missing      0
X4_missing      0
dtype: int64

In [23]:
# ── Encode X9 (district) ───────────────────────────────────────────────────
# Target encoding (> one hot encoding) — median TARGET per district on train only, then map to full dataset
train_mask = df_processed["BASE"] == "TRAIN"

# Unseen districts in test get overall median
overall_median = df_processed.loc[train_mask, "TARGET"].median()

# Smoothed target encoding — replaces the raw median map
k = 10  # smoothing factor, tune if needed

district_stats = df_processed[train_mask].groupby("X9")["TARGET"].agg(["median", "count"])
district_stats["smoothed"] = (
    (district_stats["median"] * district_stats["count"] + overall_median * k)
    / (district_stats["count"] + k)
)

df_processed["X9_encoded"] = df_processed["X9"].map(district_stats["smoothed"])
df_processed["X9_encoded"] = df_processed["X9_encoded"].fillna(overall_median)
df_processed = df_processed.drop(columns=["X9"])

In [24]:
# ── Standardization (fit on train only) ───────────────────────────────────
from sklearn.preprocessing import StandardScaler

cols_to_scale = ["X1", "X2", "X3", "X4", "X7", "X10", "X11", "X12", "X9_encoded"]

scaler = StandardScaler()
df_processed.loc[train_mask, cols_to_scale] = scaler.fit_transform(df_processed[train_mask][cols_to_scale])
df_processed.loc[~train_mask, cols_to_scale] = scaler.transform(df_processed[~train_mask][cols_to_scale])

In [25]:
df_processed.head()

,ID,TARGET,X1,X2,X3,X4,X6,X7,X8,X10,X11,X12,BASE,X3_missing,X4_missing,X9_encoded
0,6290921,8.944681,0.502630,-0.715858,-0.389180,0.087068,3.0,0.282863,3.0,0.205535,-0.047957,-1.753999,TRAIN,0,1,-0.497239
1,621369,8.709960,-0.266376,0.157672,-0.273531,1.538663,2.0,0.013725,2.0,0.838172,0.135032,0.886872,TRAIN,0,0,-0.698069
2,218437,8.850518,0.737259,0.751763,0.904066,-0.994911,4.0,-1.009001,2.0,0.743300,-0.058082,0.158356,TRAIN,0,0,-1.241424
3,3089628,8.324336,0.190979,1.049474,-0.608619,0.087068,1.0,-1.085898,1.0,0.322582,0.364048,0.067292,TRAIN,0,1,-1.736713
4,2351873,8.954544,0.010694,-1.056125,0.628709,-0.907438,1.0,0.613519,1.0,0.313685,-0.870620,-1.207612,TRAIN,0,0,-0.950441


In [26]:
# ── Save processed dataset ─────────────────────────────────────────────────
df_processed.to_csv("../data/processed/data_processed.csv", index=False)

In [27]:
# ── Train / test split ─────────────────────────────────────────────────────
train = df_processed[df_processed["BASE"] == "TRAIN"].drop(columns=["BASE"])
test  = df_processed[df_processed["BASE"] == "TEST"].drop(columns=["BASE"])

train.to_csv("../data/splits/train.csv", index=False)
test.to_csv("../data/splits/test.csv",  index=False)

print(f"Train: {train.shape} | Test: {test.shape}")

Train: (33334, 15) | Test: (16667, 15)


In [28]:
df_processed.describe()

,ID,TARGET,X1,X2,X3,X4,X6,X7,X8,X10,X11,X12,X3_missing,X4_missing,X9_encoded
count,5.000100e+04,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000,50001.000000
mean,2.709827e+06,8.831549,0.001270,0.002500,-0.001374,-0.002278,1.848443,-0.001190,1.703186,0.003679,-0.003813,-0.007039,0.398432,0.300134,-0.001713
std,2.227480e+06,0.651955,0.996558,0.994189,1.004648,0.999879,1.061108,1.007097,0.829674,0.997017,0.998406,0.997364,0.489580,0.458321,0.997946
min,9.000000e+00,6.908755,-7.308323,-3.754693,-7.555125,-3.905840,1.000000,-5.953456,1.000000,-2.092372,-4.461482,-2.118257,0.000000,0.000000,-3.348019
25%,1.084482e+06,8.437934,-0.412157,-0.584327,-0.194441,-0.357419,1.000000,-0.140069,1.000000,0.043057,-0.212242,-0.752289,0.000000,0.000000,-0.757907
50%,2.182781e+06,8.848940,0.225379,0.064587,0.028679,0.087068,2.000000,0.290553,2.000000,0.433550,-0.018202,-0.114837,0.000000,0.000000,-0.109586
75%,3.482249e+06,9.253974,0.642373,0.555740,0.258557,0.557333,2.000000,0.498174,2.000000,0.619852,0.364048,0.522614,1.000000,1.000000,0.909326
max,2.059424e+07,10.596660,1.573919,2.634221,3.029193,2.358169,4.000000,1.697761,5.000000,1.075477,3.456891,4.711583,1.000000,1.000000,2.135851
